In [12]:
import numpy as np
import pandas as pd
from scipy.stats import norm

In [13]:
def norm_cdf(x: np.ndarray) -> np.ndarray:
    return norm.cdf(x)
def norm_ppf(x: np.ndarray) -> np.ndarray:
    return norm.ppf(x)
def add_mu_hist(
    df: pd.DataFrame,
    ret_col: str = "msci_ret_weekly",
    country_col: str = "country_clean",
    date_col: str = "date",
    window_weeks: int = 52,
    annualize: int = 52,
    min_periods: int = 10,
) -> pd.DataFrame:
    """
    Adds mu_hist: trailing mean of weekly returns * annualize, shifted by 1 to avoid look-ahead.
    """
    out = df.sort_values([country_col, date_col]).copy()
    out["mu_hist"] = (
        out.groupby(country_col)[ret_col]
        .apply(lambda s: s.rolling(window_weeks, min_periods=min_periods).mean().shift(1) * annualize)
        .reset_index(level=0, drop=True)
    )
    out["mu_hist"] = out["mu_hist"].fillna(0.0)
    return out


def closed_form_pd_dd_merton_gbm(
    V0: np.ndarray,
    B0: np.ndarray,
    sigma_ann: np.ndarray,
    mu_ann: np.ndarray,
    T_years: float = 5.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Closed-form Merton (GBM) terminal default:
      DD = [ln(V0/B0) + (mu - 0.5 sigma^2) T] / (sigma sqrt(T))
      PD = Phi(-DD)
    """
    V0 = np.asarray(V0, dtype=float)
    B0 = np.asarray(B0, dtype=float)
    sigma = np.asarray(sigma_ann, dtype=float)
    mu = np.asarray(mu_ann, dtype=float)

    if np.any(V0 <= 0) or np.any(B0 <= 0):
        raise ValueError("V0 and B0 must be strictly positive to take logs.")

    denom = sigma * np.sqrt(T_years)

    # Handle sigma=0 edge case: deterministic terminal value
    dd = np.empty_like(V0, dtype=float)
    pd = np.empty_like(V0, dtype=float)

    zero_sigma = denom == 0
    nonzero = ~zero_sigma

    # Nonzero sigma: standard formula
    dd[nonzero] = (
        (np.log(V0[nonzero] / B0[nonzero]) + (mu[nonzero] - 0.5 * sigma[nonzero]**2) * T_years)
        / denom[nonzero]
    )
    pd[nonzero] = norm_cdf(-dd[nonzero])

    # Zero sigma: compare deterministic terminal value to barrier
    if np.any(zero_sigma):
        VT_det = V0[zero_sigma] * np.exp(mu[zero_sigma] * T_years)
        pd[zero_sigma] = (VT_det < B0[zero_sigma]).astype(float)
        # DD is +/- inf in theory; we set large magnitude values for practicality
        dd[zero_sigma] = np.where(pd[zero_sigma] > 0, -1e6, 1e6)

    return pd, dd


def mc_pd_dd_gbm_terminal(
    V0: np.ndarray,
    B0: np.ndarray,
    sigma_ann: np.ndarray,
    mu_ann: np.ndarray,
    T_years: float = 5.0,
    n_paths: int = 20000,
    seed: int = 7,
    clip_pd: float = 1e-10,
    chunk_rows: int = 300,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Monte Carlo PD and equivalent-normal DD for terminal-default Merton-style GBM.
    Uses exact terminal distribution simulation (no time stepping).
    """
    V0 = np.asarray(V0, dtype=float)
    B0 = np.asarray(B0, dtype=float)
    sigma_ann = np.asarray(sigma_ann, dtype=float)
    mu_ann = np.asarray(mu_ann, dtype=float)

    if np.any(V0 <= 0) or np.any(B0 <= 0):
        raise ValueError("V0 and B0 must be strictly positive to take logs.")

    rng = np.random.default_rng(seed)
    n = V0.shape[0]

    logV0 = np.log(V0)
    logB0 = np.log(B0)

    drift = (mu_ann - 0.5 * sigma_ann**2) * T_years
    volT = sigma_ann * np.sqrt(T_years)

    pd_hat = np.empty(n, dtype=float)

    for start in range(0, n, chunk_rows):
        end = min(n, start + chunk_rows)
        m = end - start

        Z = rng.standard_normal(size=(m, n_paths))
        logVT = logV0[start:end, None] + drift[start:end, None] + volT[start:end, None] * Z
        pd_hat[start:end] = (logVT < logB0[start:end, None]).mean(axis=1)

    se_pd = np.sqrt(pd_hat * (1.0 - pd_hat) / n_paths)

    # Equivalent-normal DD from PD (clip to avoid inf)
    pd_clip = np.clip(pd_hat, clip_pd, 1.0 - clip_pd)
    dd_hat = norm_ppf(1.0 - pd_clip)

    return pd_hat, dd_hat, se_pd


def run_gbm_mc_on_panel(
    panel_csv_path: str,
    out_csv_path: str,
    mu_mode: str = "zero",        # "zero" or "hist"
    n_paths: int = 20000,
    seed: int = 7,
    T_years: float = 5.0,
    use_scaled_V0: bool = True,   # V0 = scale_mult * barrier
    scale_mult: float = 1.5,
) -> pd.DataFrame:
    """
    Loads panel, computes (optional) historical drift, runs GBM MC + closed form,
    and saves results.

    Expected columns:
      - country_clean, date
      - msci_index (only if use_scaled_V0=False)
      - default_barrier
      - msci_vol_52w (annualized)
      - msci_ret_weekly (only if mu_mode='hist')
    """
    df = pd.read_csv(panel_csv_path, parse_dates=["date"])

    if mu_mode not in {"zero", "hist"}:
        raise ValueError("mu_mode must be 'zero' or 'hist'.")

    if mu_mode == "hist":
        if "msci_ret_weekly" not in df.columns:
            raise ValueError("msci_ret_weekly missing; required for mu_mode='hist'.")
        df = add_mu_hist(df)
        mu = df["mu_hist"].to_numpy(dtype=float)
    else:
        df["mu_hist"] = 0.0
        mu = np.zeros(len(df), dtype=float)

    B0 = df["default_barrier"].to_numpy(dtype=float)
    sigma = df["msci_vol_52w"].to_numpy(dtype=float)  # annualized

    # Use your scaling to ensure units match barrier
    if use_scaled_V0:
        V0 = scale_mult * B0
        df["V0_used"] = V0
        df["V0_mode"] = f"scaled_{scale_mult}x_barrier"
    else:
        V0 = df["msci_index"].to_numpy(dtype=float)
        df["V0_used"] = V0
        df["V0_mode"] = "raw_msci_index"

    # Closed form
    pd_cf, dd_cf = closed_form_pd_dd_merton_gbm(
        V0=V0, B0=B0, sigma_ann=sigma, mu_ann=mu, T_years=T_years
    )
    df["pd_cf_gbm"] = pd_cf
    df["dd_cf_gbm"] = dd_cf

    # Monte Carlo
    pd_mc, dd_mc, se_mc = mc_pd_dd_gbm_terminal(
        V0=V0, B0=B0, sigma_ann=sigma, mu_ann=mu, T_years=T_years, n_paths=n_paths, seed=seed
    )
    df["pd_mc_gbm"] = pd_mc
    df["dd_mc_gbm"] = dd_mc
    df["pd_mc_se"] = se_mc

    # Helpful comparison columns
    df["pd_abs_diff"] = np.abs(df["pd_cf_gbm"] - df["pd_mc_gbm"])
    df["dd_abs_diff"] = np.abs(df["dd_cf_gbm"] - df["dd_mc_gbm"])

    df.to_csv(out_csv_path, index=False)
    return df



In [14]:
in_path = "data/processed/merton_panel.csv"
out_path =  "output/mc_gbm_results.csv"


df_res = run_gbm_mc_on_panel(
        panel_csv_path=in_path,
        out_csv_path=out_path,
        mu_mode="zero",
        n_paths=1000,
        seed=7,
        T_years=5.0,
    )


In [15]:
df_res

,country,country_clean,group,date,msci_index,msci_ret_weekly,msci_vol_52w,n_trading_days,debt_st,debt_lt,...,mu_hist,V0_used,V0_mode,pd_cf_gbm,dd_cf_gbm,pd_mc_gbm,dd_mc_gbm,pd_mc_se,pd_abs_diff,dd_abs_diff
0,Brazil,brazil,oil_exporter,2014-01-03,804.695,-0.023366,0.220537,5,0.0,5.867838e+10,...,0.0,4.400879e+10,scaled_1.5x_barrier,0.282425,0.575652,0.290,0.553385,0.014349,0.007575,0.022267
1,Brazil,brazil,oil_exporter,2014-01-10,788.350,-0.020521,0.220925,5,0.0,5.867838e+10,...,0.0,4.400879e+10,scaled_1.5x_barrier,0.283060,0.573775,0.304,0.512930,0.014546,0.020940,0.060844
2,Brazil,brazil,oil_exporter,2014-01-17,780.773,-0.009658,0.220657,5,0.0,5.867838e+10,...,0.0,4.400879e+10,scaled_1.5x_barrier,0.282622,0.575070,0.302,0.518657,0.014519,0.019378,0.056413
3,Brazil,brazil,oil_exporter,2014-01-24,743.312,-0.049169,0.224817,5,0.0,5.867838e+10,...,0.0,4.400879e+10,scaled_1.5x_barrier,0.289374,0.555213,0.277,0.591777,0.014152,0.012374,0.036564
4,Brazil,brazil,oil_exporter,2014-01-31,737.863,-0.007358,0.223771,5,0.0,5.867838e+10,...,0.0,4.400879e+10,scaled_1.5x_barrier,0.287687,0.560154,0.288,0.559237,0.014320,0.000313,0.000917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10956,Turkey,turkey,control,2024-11-29,609.507,0.006911,0.207643,5,0.0,1.216690e+11,...,0.0,9.125175e+10,scaled_1.5x_barrier,0.260721,0.641124,0.274,0.600760,0.014104,0.013279,0.040364
10957,Turkey,turkey,control,2024-12-06,634.388,0.040010,0.210889,5,0.0,1.216690e+11,...,0.0,9.125175e+10,scaled_1.5x_barrier,0.266296,0.624054,0.259,0.646431,0.013853,0.007296,0.022377
10958,Turkey,turkey,control,2024-12-13,634.804,0.000656,0.210808,5,0.0,1.216690e+11,...,0.0,9.125175e+10,scaled_1.5x_barrier,0.266158,0.624475,0.279,0.585815,0.014183,0.012842,0.038660
10959,Turkey,turkey,control,2024-12-20,605.698,-0.046935,0.206713,5,0.0,1.216690e+11,...,0.0,9.125175e+10,scaled_1.5x_barrier,0.259110,0.646091,0.264,0.631062,0.013939,0.004890,0.015029


In [8]:
import numpy as np
import pandas as pd

df = pd.read_csv(in_path, parse_dates=["date"])

x = np.log(df["msci_index"].values) - np.log(df["default_barrier"].values)
print("log(V0/B0) summary:")
print(pd.Series(x).describe())

print("Min V0:", df["msci_index"].min(), "Max V0:", df["msci_index"].max())
print("Min B0:", df["default_barrier"].min(), "Max B0:", df["default_barrier"].max())


log(V0/B0) summary:
count    10961.000000
mean       -17.615674
std          1.374340
min        -22.439073
25%        -18.323347
50%        -17.463099
75%        -16.875092
max        -14.732842
dtype: float64
Min V0: 26.64 Max V0: 2730.065
Min B0: 2669493143.5 Max B0: 258345143164.0
